# Cycling Safety Perception Maps: London

Self-contained notebook. Loads data, aggregates, and produces:

**Part A: Interactive HTML maps:**
- GPT safety score map (colour + sigma size)
- Human safety score map (colour + sigma size)
- GPT vs Human disagreement map

**Part B: Static publication maps (PDF, 300 DPI):**
- GPT safety score: 2x2 mosaic (London overview + 3 zoom levels)
- Human safety score: same layout

**Design choices:**
- Transparency: 0.5 for all dots
- Circle size: **large = low sigma (certain), small = high sigma (uncertain)**
- Colour: RdYlGn - Red=unsafe, Green=safe

Run cells in order from top to bottom.

## 1. Install dependencies

In [ ]:
import sys, subprocess
for pkg in ['folium', 'contextily', 'geopandas', 'pyproj',
            'shapely', 'matplotlib-scalebar']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)
print('Done.')

## 2. Imports

In [ ]:
import os, json, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.patches import PathPatch
from matplotlib.path import Path
import folium
import contextily as ctx
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union
from scipy.stats import pearsonr, spearmanr
from matplotlib_scalebar.scalebar import ScaleBar
warnings.filterwarnings('ignore')
print('Imports loaded.')

## 3. Configuration

**Key design parameters:**
- `POINT_ALPHA = 0.5` - transparency for all dots (reviewer comment)
- `SIZE_MIN / SIZE_MAX` - **low sigma = large dot, high sigma = small dot** (reviewer comment)
- `ZOOM_LEVELS` - 4 zoom boxes for the 2x2 mosaic static maps

In [ ]:
import numpy as np

# ── PATHS ────────────────────────────────────────────────────────────────────
COORDS_PATH     = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'
GPT_RANK_PATH   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\gpt_trueskill_rankings_200k_20_06_26.csv'
HUMAN_RANK_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\human_trueskill_rankings_final_20_06_26.csv'
OUTPUT_DIR      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── COLOUR MAPS ───────────────────────────────────────────────────────────────
CMAP_SCORE    = 'RdYlGn'
CMAP_DISAGREE = 'PuRd'
FOLIUM_SCORE  = 'RdYlGn'

# ── BIN SETTINGS ─────────────────────────────────────────────────────────────
BIN_COLORS  = ['#b2182b', '#e08214', '#1a7837']
BIN_LABELS  = ['Low safety', 'Medium safety', 'High safety']
BIN_MARKERS = ['o', 'D', '*']
BIN_SIZES   = [1.2, 1.2, 1.6]

# ── STATIC MAP SETTINGS ───────────────────────────────────────────────────────
DPI         = 300
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

POINT_ALPHA  = 0.9
SIZE_MIN     = 2
SIZE_MAX     = 12
FOLIUM_R_MIN = 1
FOLIUM_R_MAX = 5

# ── 4-LEVEL ZOOM BOXES ────────────────────────────────────────────────────────
CENTRE_LAT  = 51.5074
CENTRE_LON  = -0.1278

# Holborn centre for the fourth panel
HOLBORN_LAT = 51.5174
HOLBORN_LON = -0.1200

lat           = 51.5
lon_km        = np.cos(np.radians(lat)) * 111.32
lat_km        = 111.32
target_aspect = (18/2 * 0.85) / (14/2 * 0.85)

HALF_WIDTHS_LON = [0.50, 0.20, 0.10, 0.045]
zoom_labels     = ['Greater London', 'Inner London', 'Central London', 'Holborn']

ZOOM_LEVELS = []
for i, (label, half_lon) in enumerate(zip(zoom_labels, HALF_WIDTHS_LON)):
    half_lat   = half_lon * lon_km / (lat_km * target_aspect)
    centre_lat = HOLBORN_LAT if i == 3 else CENTRE_LAT
    centre_lon = HOLBORN_LON if i == 3 else CENTRE_LON
    ZOOM_LEVELS.append({
        'label'  : label,
        'lon_min': centre_lon - half_lon,
        'lon_max': centre_lon + half_lon,
        'lat_min': centre_lat - half_lat,
        'lat_max': centre_lat + half_lat,
    })

print('Configuration loaded.')
print(f'  Output     : {OUTPUT_DIR}')
print(f'  Alpha      : {POINT_ALPHA}')
print(f'  Size range : {SIZE_MIN} to {SIZE_MAX}')
print(f'  Zoom levels: {len(ZOOM_LEVELS)}')
for z in ZOOM_LEVELS:
    print(f"    {z['label']:<16} lon [{z['lon_min']:.3f}, {z['lon_max']:.3f}]  "
          f"lat [{z['lat_min']:.4f}, {z['lat_max']:.4f}]")

In [ ]:
# ── Helper functions for London boundary masking ──────────────────────────────
import io as _io

def geom_to_path(geom):
    """Convert a shapely geometry to a matplotlib Path for boundary masking."""
    def ring_to_verts(ring):
        coords = np.array(ring.coords)
        return coords, [Path.MOVETO] + [Path.LINETO] * (len(coords) - 2) + [Path.CLOSEPOLY]
    verts, codes = [], []
    geoms = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for g in geoms:
        v, c = ring_to_verts(g.exterior)
        verts.extend(v); codes.extend(c)
        for interior in g.interiors:
            v, c = ring_to_verts(interior)
            verts.extend(v); codes.extend(c)
    return Path(np.array(verts), codes)


def load_london_boundary():
    """Load and dissolve the London borough boundaries into a single polygon."""
    try:
        url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
                   'master/json/administrative/eng/lad.json')
        resp    = requests.get(url, timeout=20)
        gdf_raw = gpd.read_file(_io.BytesIO(resp.content))
        london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].dissolve().to_crs('EPSG:3857')
        print('London boundary loaded.')
        return london
    except Exception as e:
        print(f'Boundary failed: {e}')
        return None


london_boundary = load_london_boundary()
print('Helper functions defined.')

## 4. Load data

In [ ]:
# ── load corrected coordinates ────────────────────────────────────────────────
coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

point_coords = (
    coords_df.groupby('point_id')[['actual_lat', 'actual_lng']]
    .first().reset_index()
    .rename(columns={'actual_lat': 'lat', 'actual_lng': 'lng'})
)
print(f'Unique locations with corrected coordinates : {len(point_coords):,}')

# ── exclude boat-captured river locations ────────────────────────────────────
# identified by checking points close to the official OS Open Rivers Thames
# geometry, then visually confirming each one against the actual Street View
# photo. 11 locations were confirmed as boat-based captures on the water with
# no cycling relevance. One nearby location (2187) is on a bridge and was kept
# since bridges are legitimate cyclable infrastructure.
RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]

n_before = len(point_coords)
point_coords = point_coords[
    ~point_coords['point_id'].isin(RIVER_EXCLUDE_IDS)
].reset_index(drop=True)
n_after = len(point_coords)

print(f'Excluded boat-captured river locations      : {n_before - n_after}')
print(f'Locations remaining                          : {n_after:,}')

# ── load rankings ─────────────────────────────────────────────────────────────
gpt_df   = pd.read_csv(GPT_RANK_PATH)
human_df = pd.read_csv(HUMAN_RANK_PATH)
print(f'\nGPT rankings   : {len(gpt_df):,} rows')
print(f'Human rankings : {len(human_df):,} rows')

## 5. Aggregate 4 headings per location

Sigma-weighted mean of TrueSkill mu.
**Note on sigma encoding:** low sigma = more certain = larger dot.

In [ ]:
def normalise_01(series):
    mn, mx = series.min(), series.max()
    if mx > mn:
        return (series - mn) / (mx - mn)
    return pd.Series(np.zeros(len(series)), index=series.index)


def aggregate_location_scores(rank_df, mu_col, sigma_col, label):
    df = rank_df.copy()
    df['image_name'] = df['image_name'].apply(
        lambda x: os.path.basename(str(x).strip())
    )
    df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')
    merged = df.merge(point_coords, on='point_id', how='inner')
    print(f'  {label}: {merged["point_id"].nunique():,} unique locations matched')

    def weighted_agg(grp):
        mu    = grp[mu_col].values
        sigma = grp[sigma_col].values
        w     = 1.0 / np.maximum(sigma ** 2, 1e-12)
        return pd.Series({
            'score_weighted': np.sum(w * mu) / np.sum(w),
            'score_mean'    : mu.mean(),
            'sigma_mean'    : sigma.mean(),
            'n_headings'    : len(grp),
            'lat'           : grp['lat'].iloc[0],
            'lng'           : grp['lng'].iloc[0],
        })

    agg = merged.groupby('point_id').apply(weighted_agg).reset_index()
    agg['score_norm']     = normalise_01(agg['score_weighted'])
    agg['sigma_norm']     = normalise_01(agg['sigma_mean'])
    agg['certainty_norm'] = 1.0 - agg['sigma_norm']
    return agg


def assign_bins(agg_df, score_col='score_weighted'):
    df = agg_df.copy()
    df['bin'] = pd.qcut(
        df[score_col], q=3, labels=[0, 1, 2], duplicates='drop'
    ).astype(int)
    return df


def certainty_to_size(certainty_norm_series):
    return SIZE_MIN + certainty_norm_series * (SIZE_MAX - SIZE_MIN)


def to_gdf(df):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df['lng'], df['lat'])],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return gdf


gpt_agg   = aggregate_location_scores(gpt_df,   'gpt_mu',   'gpt_sigma',   'GPT')
human_agg = aggregate_location_scores(human_df, 'human_mu', 'human_sigma', 'Human')

gpt_plot   = assign_bins(gpt_agg)
human_plot = assign_bins(human_agg)

for df in [gpt_plot, human_plot]:
    df['sigma_norm']     = normalise_01(df['sigma_mean'])
    df['certainty_norm'] = 1.0 - df['sigma_norm']
    df['dot_size']       = certainty_to_size(df['certainty_norm'])

gpt_gdf   = to_gdf(gpt_plot)
human_gdf = to_gdf(human_plot)

disagree_df = gpt_agg[['point_id', 'lat', 'lng', 'score_norm']].merge(
    human_agg[['point_id', 'score_norm']], on='point_id', suffixes=('_gpt', '_human')
)
disagree_df['abs_diff'] = (
    disagree_df['score_norm_gpt'] - disagree_df['score_norm_human']
).abs()
disagree_df['diff_norm'] = normalise_01(disagree_df['abs_diff'])

rho_s, _ = spearmanr(disagree_df['score_norm_gpt'], disagree_df['score_norm_human'])
rho_p, _ = pearsonr(disagree_df['score_norm_gpt'],  disagree_df['score_norm_human'])
print(f'\nGPT vs Human: Spearman rho={rho_s:.4f}  Pearson r={rho_p:.4f}')
print(f'Overlapping locations: {len(disagree_df):,}')

for label, df in [('GPT', gpt_plot), ('Human', human_plot)]:
    counts = df['bin'].value_counts().sort_index()
    print(f'\n{label} bin distribution:')
    for b, lbl in enumerate(BIN_LABELS):
        n = counts.get(b, 0)
        print(f'  Bin {b} ({lbl}): {n:,}  ({n/len(df)*100:.1f}%)')

# Static publication maps (PDF)

**2x2 mosaic layout per source (reviewer comment):**
- Top left: Greater London overview
- Top right: Inner London
- Bottom left: City of London
- Bottom right: Road level (Bank area)

**Encoding:**
- Colour = 3-bin safety class (Red / Yellow / Green)
- Circle size = certainty - **large = low sigma (certain)**
- Alpha = 0.5

Requires internet for basemap tiles. Tiles cached after first run.

In [ ]:
def assign_bins(agg_df, score_col='score_weighted'):
    df = agg_df.copy()
    df['bin'] = pd.qcut(
        df[score_col], q=3, labels=[0, 1, 2], duplicates='drop'
    ).astype(int)
    return df


def certainty_to_size(certainty_norm_series):
    """INVERTED: high certainty (low sigma) = large dot."""
    return SIZE_MIN + certainty_norm_series * (SIZE_MAX - SIZE_MIN)


def to_gdf(df):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df['lng'], df['lat'])],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return gdf


gpt_plot   = assign_bins(gpt_agg)
human_plot = assign_bins(human_agg)

for df in [gpt_plot, human_plot]:
    df['sigma_norm']     = normalise_01(df['sigma_mean'])
    df['certainty_norm'] = 1.0 - df['sigma_norm']
    df['dot_size']       = certainty_to_size(df['certainty_norm'])

gpt_gdf   = to_gdf(gpt_plot)
human_gdf = to_gdf(human_plot)

# updated colours — visible on white basemap
BIN_COLORS  = ['#d73027', '#f4a42c', '#1a9850']
BIN_LABELS  = ['Low safety', 'Medium safety', 'High safety']

for label, df in [('GPT', gpt_plot), ('Human', human_plot)]:
    counts = df['bin'].value_counts().sort_index()
    print(f'{label} bin distribution:')
    for b, lbl in enumerate(BIN_LABELS):
        n = counts.get(b, 0)
        print(f'  Bin {b} ({lbl}): {n:,}  ({n/len(df)*100:.1f}%)')
    print()

In [ ]:
def zoom_to_mercator(zoom):
    corners = gpd.GeoDataFrame(
        geometry=[
            Point(zoom['lon_min'], zoom['lat_min']),
            Point(zoom['lon_max'], zoom['lat_max'])
        ],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return {
        'label' : zoom['label'],
        'x_min' : corners.geometry.iloc[0].x,
        'y_min' : corners.geometry.iloc[0].y,
        'x_max' : corners.geometry.iloc[1].x,
        'y_max' : corners.geometry.iloc[1].y,
    }


zoom_boxes = [zoom_to_mercator(z) for z in ZOOM_LEVELS]

print('Zoom boxes in Web Mercator:')
for z in zoom_boxes:
    print(f'  {z["label"]}')

In [ ]:
import io as _io
from shapely.geometry import box as shapely_box

basemap_per_panel = [
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.PositronNoLabels,
]
size_scale_per_panel = [0.8, 1.5, 3.0, 6.0]

london_boundary = None
print('Loading London boundary...')
try:
    url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
               'master/json/administrative/eng/lad.json')
    resp    = requests.get(url, timeout=20)
    gdf_raw = gpd.read_file(_io.BytesIO(resp.content))
    london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].copy()
    print(f'  London boroughs found: {len(london)}')
    london_boundary = london.dissolve().to_crs('EPSG:3857')
    print(f'  Bounds: {[round(x) for x in london_boundary.total_bounds]}')
except Exception as e:
    print(f'  Failed: {e}')
    from shapely.geometry import MultiPoint
    pts             = MultiPoint(list(zip(gpt_gdf.geometry.x, gpt_gdf.geometry.y)))
    hull            = pts.convex_hull.buffer(3000)
    london_boundary = gpd.GeoDataFrame(geometry=[hull], crs='EPSG:3857')
    print('  Fallback boundary created.')


def add_london_border(ax, boundary):
    if boundary is None:
        return
    try:
        boundary.boundary.plot(
            ax=ax, color='#333333',
            linewidth=1.2, alpha=1.0, zorder=7
        )
    except Exception as e:
        print(f'  Border failed: {e}')


print('Setup complete.')

In [ ]:
from shapely.ops import unary_union

def draw_panel(ax, gdf, title, basemap_source,
               xlim=None, ylim=None, add_scalebar=False,
               size_scale=1.0, show_border=True,
               apply_mask=False):

    ax.set_facecolor('white')
    ax.set_aspect('auto')
    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    ctx.add_basemap(
        ax, crs=gdf.crs,
        source=basemap_source, zoom='auto', alpha=0.9
    )

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    if apply_mask and london_boundary is not None:
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        buffer     = 5000
        extent     = shapely_box(x0 - buffer, y0 - buffer,
                                 x1 + buffer, y1 + buffer)
        london_u   = unary_union(london_boundary.geometry.values)
        mask_geom  = extent.difference(london_u)
        mask_patch = PathPatch(geom_to_path(mask_geom),
                               fc='white', ec='none', alpha=1.0, zorder=4)
        ax.add_patch(mask_patch)

    for b, (color, marker, msize_factor) in enumerate(
        zip(BIN_COLORS, BIN_MARKERS, BIN_SIZES)
    ):
        subset = gdf[gdf['bin'] == b]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            color=color, marker=marker,
            s=subset['dot_size'] * size_scale * msize_factor,
            alpha=POINT_ALPHA,
            linewidths=0, rasterized=True, zorder=5
        )

    if show_border:
        add_london_border(ax, london_boundary)

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    if add_scalebar:
        ax.add_artist(
            ScaleBar(1, units='m', location='lower right',
                     frameon=True, color='black', box_alpha=0.7,
                     font_properties={'size': 7, 'family': FONT_FAMILY})
        )

    ax.annotate(
        'N', xy=(0.04, 0.12), xycoords='axes fraction',
        fontsize=8, fontfamily=FONT_FAMILY, fontweight='bold',
        ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='grey', alpha=0.8),
        zorder=8
    )
    ax.annotate(
        '', xy=(0.04, 0.12), xycoords='axes fraction',
        xytext=(0.04, 0.06), textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
        zorder=8
    )

    ax.set_title(title, fontsize=13, fontfamily=FONT_FAMILY,
                 fontweight='bold', pad=6)

    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)


print('draw_panel defined.')

In [ ]:
import copy

CARTO_API_KEY = '...'

def get_carto_basemap():
    provider = copy.deepcopy(ctx.providers.CartoDB.Positron)
    provider['url'] = provider['url'] + f'?key={CARTO_API_KEY}'
    return provider

basemap_per_panel = [get_carto_basemap() for _ in range(4)]

## checking problems visually

In [ ]:
import io
from PIL import Image
from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    '...'
    '...'
    '...'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def fetch_image(image_name):
    blob_data = container_client.download_blob(
        IMAGE_PREFIX + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')

# all 12 candidates checked, sorted by distance to the Thames
candidate_ids = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '2187', '4943', '4354', '1610', '3822', '1984'
]
decisions = {
    '2705': 'Excluded — boat on river',
    '2138': 'Excluded — boat on river',
    '4577': 'Excluded — boat on river',
    '395' : 'Excluded — boat on river',
    '2028': 'Excluded — boat on river',
    '2858': 'Excluded — boat on river',
    '2187': 'Kept — on bridge',
    '4943': 'Excluded — boat on river',
    '4354': 'Excluded — boat on river',
    '1610': 'Excluded — boat on river',
    '3822': 'Excluded — boat on river',
    '1984': 'Excluded — boat on river',
}

fig, axes = plt.subplots(len(candidate_ids), 4, figsize=(16, 4 * len(candidate_ids)))
fig.patch.set_facecolor('white')

for row, pid in enumerate(candidate_ids):
    decision = decisions[pid]
    colour   = '#c0392b' if 'Excluded' in decision else '#1a9850'

    for col, heading in enumerate([0, 90, 180, 270]):
        ax = axes[row, col]
        ax.set_facecolor('white')
        img_name = f'{pid}_{heading}.jpg'
        try:
            img = fetch_image(img_name)
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Not found', ha='center', va='center',
                     transform=ax.transAxes, fontsize=8, color='red')

        if col == 0:
            ax.set_ylabel(f'Point {pid}\n{decision}', fontsize=9,
                           color=colour, fontweight='bold')
        ax.set_title(f'{heading}\u00b0', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'river_exclusion_verification.pdf')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# Human static map

In [ ]:
print('Building Human 2x2 mosaic map...')

human_gdf_plot = human_gdf.copy()

fig_human, axes_human = plt.subplots(2, 2, figsize=(18, 14),
                                      gridspec_kw={'wspace': 0, 'hspace': 0.05})
fig_human.patch.set_facecolor('white')
fig_human.suptitle(
    f'Human Cycling Safety Score — London  (n = {len(human_gdf_plot):,} locations)\n'
    'Colour and shape: 3-bin quantile class  |  \u03b1 = 0.8',
    fontsize=13, fontfamily=FONT_FAMILY, fontweight='bold', y=1.01
)

for idx, ((row, col), zbox) in enumerate(zip(panel_positions, zoom_boxes)):
    ax = axes_human[row, col]
    draw_panel(
        ax, human_gdf_plot, zbox['label'],
        basemap_source=basemap_per_panel[idx],
        xlim=(zbox['x_min'], zbox['x_max']),
        ylim=(zbox['y_min'], zbox['y_max']),
        add_scalebar=True,
        size_scale=size_scale_per_panel[idx],
        show_border=True,
        apply_mask=(idx == 0)
    )

fig_human.legend(
    handles=shape_handles,
    loc='lower center', ncol=3, fontsize=12,
    frameon=True, framealpha=0.95,
    prop={'family': FONT_FAMILY, 'size': 12},
    bbox_to_anchor=(0.5, 0.01),
    title='Safety class (colour and shape)',
    title_fontproperties={
        'family': FONT_FAMILY, 'size': 12, 'weight': 'bold'
    },
    edgecolor='#cccccc'
)

fig_human.tight_layout(pad=0.5)
out_human = os.path.join(OUTPUT_DIR, 'static_human_score_mosaic.pdf')
fig_human.savefig(out_human, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_human}')

In [ ]:
print('=' * 60)
print('ALL FILES SAVED')
print('=' * 60)
print(f'  Output folder: {OUTPUT_DIR}')
print()
print('  HTML maps (open in browser):')
print('    map_gpt_score.html')
print('    map_human_score.html')
print('    map_disagreement.html')
print()
print('  Static PDFs (for paper):')
print('    static_gpt_score_mosaic.pdf   (2x2: Greater London / Inner / City / Road level)')
print('    static_human_score_mosaic.pdf (same layout)')
print()
print(f'  GPT locations   : {len(gpt_agg):,}')
print(f'  Human locations : {len(human_agg):,}')
print(f'  Spearman rho    : {rho_s:.4f}')
print(f'  Pearson r       : {rho_p:.4f}')

# New Article Figures 11 and 12

In [ ]:
# ── Combined continuous safety score map: Human and GPT side by side ──────────
# 2 rows (Human, GPT) x 3 columns (Greater London, Central London, Holborn)

import copy
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import contextily as ctx
import numpy as np
import os
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union
from matplotlib.patches import PathPatch
from matplotlib.path import Path

mpl.rcParams['patch.antialiased'] = False

CARTO_API_KEY = '...'

def get_carto_basemap(base_provider):
    provider = copy.deepcopy(base_provider)
    provider['url'] = provider['url'] + f'?key={CARTO_API_KEY}'
    return provider

# ── Settings ──────────────────────────────────────────────────────────────────
SELECTED_ZOOM_LABELS = ['Greater London', 'Central London', 'Holborn']
CMAP_CONTINUOUS      = plt.cm.RdYlGn
POINT_SIZE           = 18
POINT_ALPHA          = 0.85
FONT_FAMILY          = 'DejaVu Sans'
DPI                  = 600
OUTPUT_DIR           = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams['font.family'] = FONT_FAMILY

selected_boxes = [z for z in zoom_boxes if z['label'] in SELECTED_ZOOM_LABELS]
selected_boxes = sorted(selected_boxes,
                        key=lambda z: SELECTED_ZOOM_LABELS.index(z['label']))

basemaps = [
    get_carto_basemap(ctx.providers.CartoDB.Positron),
    get_carto_basemap(ctx.providers.CartoDB.Positron),
    get_carto_basemap(ctx.providers.CartoDB.PositronNoLabels),
]

rows = [
    ('Human crowdsourced votes', human_gdf, 'score_norm'),
    ('GPT-5.2 synthetic votes',  gpt_gdf,   'score_norm'),
]

norm = mcolors.Normalize(vmin=0, vmax=1)

fig = plt.figure(figsize=(24, 12))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(
    2, 1, figure=fig,
    hspace=0.08
)

axes_all = {}

for row_idx, (row_label, gdf, score_col) in enumerate(rows):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 3,
        subplot_spec=outer[row_idx],
        wspace=0.06,
        width_ratios=[1.3, 1, 1]
    )

    for col_idx, zbox in enumerate(selected_boxes):
        ax = fig.add_subplot(inner[0, col_idx])
        ax.set_facecolor('white')
        ax.patch.set_visible(False)
        ax.set_aspect('auto')
        xlim = (zbox['x_min'], zbox['x_max'])
        ylim = (zbox['y_min'], zbox['y_max'])
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # Basemap
        ctx.add_basemap(
            ax, crs=gdf.crs,
            source=basemaps[col_idx],
            zoom='auto', alpha=0.85
        )
        ax.set_aspect('auto')
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # London boundary mask + outline on Greater London panel only
        if col_idx == 0 and london_boundary is not None:
            buffer    = 5000
            extent    = shapely_box(
                xlim[0]-buffer, ylim[0]-buffer,
                xlim[1]+buffer, ylim[1]+buffer
            )
            london_u  = unary_union(london_boundary.geometry.values)
            mask_geom = extent.difference(london_u)

            def geom_to_path_local(geom):
                def ring_to_verts(ring):
                    coords = np.array(ring.coords)
                    return coords, ([Path.MOVETO] +
                                    [Path.LINETO]*(len(coords)-2) +
                                    [Path.CLOSEPOLY])
                verts, codes = [], []
                geoms = [geom] if geom.geom_type=='Polygon' else list(geom.geoms)
                for g in geoms:
                    v, c = ring_to_verts(g.exterior)
                    verts.extend(v); codes.extend(c)
                    for interior in g.interiors:
                        v, c = ring_to_verts(interior)
                        verts.extend(v); codes.extend(c)
                return Path(np.array(verts), codes)

            mask_patch = PathPatch(
                geom_to_path_local(mask_geom),
                fc='white', ec='none', alpha=1.0, zorder=4
            )
            ax.add_patch(mask_patch)

            # TEMPORARILY DISABLED to test whether the "line" is actually
            # the real London boundary outline rather than a rendering bug.
            # london_boundary.boundary.plot(
            #     ax=ax, color='#333333', linewidth=1.0, zorder=5
            # )

        # Scatter points coloured by normalised score
        colors = CMAP_CONTINUOUS(norm(gdf[score_col].values))
        ax.scatter(
            gdf.geometry.x, gdf.geometry.y,
            c=colors,
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            linewidths=0,
            rasterized=True,
            zorder=6
        )

        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        if row_idx == 0:
            ax.set_title(zbox['label'], fontsize=11,
                         fontweight='bold', pad=6,
                         fontfamily=FONT_FAMILY)

        ax.annotate('N', xy=(0.04, 0.12), xycoords='axes fraction',
                    fontsize=8, fontfamily=FONT_FAMILY, fontweight='bold',
                    ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2',
                              fc='white', ec='grey', alpha=0.8), zorder=8)
        ax.annotate('', xy=(0.04, 0.12), xycoords='axes fraction',
                    xytext=(0.04, 0.06), textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                    zorder=8)

        ax.tick_params(left=False, bottom=False,
                       labelleft=False, labelbottom=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        axes_all[(row_idx, col_idx)] = ax

# ── Row labels on the left ────────────────────────────────────────────────────
fig.canvas.draw()
for row_idx, (row_label, _, _) in enumerate(rows):
    ax0   = axes_all[(row_idx, 0)]
    pos   = ax0.get_position()
    y_mid = (pos.y0 + pos.y1) / 2
    fig.text(
        pos.x0 - 0.03, y_mid,
        row_label,
        fontsize=10, fontweight='bold',
        ha='right', va='center',
        rotation=90, fontfamily=FONT_FAMILY,
        color='#222222'
    )

# ── Shared colorbar ───────────────────────────────────────────────────────────
fig.canvas.draw()
ax_last  = axes_all[(1, 2)]
pos_last = ax_last.get_position()
ax_first = axes_all[(0, 2)]
pos_first = ax_first.get_position()

cbar_ax = fig.add_axes([
    pos_last.x1 + 0.012,
    pos_last.y0,
    0.015,
    pos_first.y1 - pos_last.y0
])
sm   = plt.cm.ScalarMappable(cmap=CMAP_CONTINUOUS, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Normalised safety score\n(0 = least safe, 1 = safest)',
               fontsize=9, fontfamily=FONT_FAMILY)
cbar.ax.tick_params(labelsize=8)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(['0.00', '0.25', '0.50', '0.75', '1.00'])

# ── Save ──────────────────────────────────────────────────────────────────────
out_png = os.path.join(OUTPUT_DIR, 'figure_combined_safety_gradient.png')
out_pdf = os.path.join(OUTPUT_DIR, 'figure_combined_safety_gradient.pdf')
fig.savefig(out_png, dpi=DPI, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved PNG: {out_png}')
print(f'Saved PDF: {out_pdf}')

# Extra Figures for the Article

In [ ]:
import io
import time
import requests as req
from PIL import Image
from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    '...'
    '...'
    '...'
)
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client('london')


def fetch_image(image_name):
    blob_data = container_client.download_blob(
        'training/training/Images/' + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')


def reverse_geocode(lat, lng):
    try:
        url    = 'https://nominatim.openstreetmap.org/reverse'
        params = {
            'lat': lat, 'lon': lng,
            'format': 'json', 'zoom': 17,
            'addressdetails': 1
        }
        headers = {'User-Agent': 'cycling-safety-research/1.0'}
        r    = req.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        addr = data.get('address', {})
        road = addr.get('road', '')
        area = (addr.get('suburb') or addr.get('neighbourhood')
                or addr.get('quarter') or addr.get('city_district', ''))
        if road and area:   return f'{road}, {area}'
        elif road:          return road
        elif area:          return area
        else:               return data.get('display_name', '').split(',')[0]
    except Exception:
        return 'Location unknown'


def get_example_images(rank_df, mu_col, sigma_col,
                        agg_df, n_top=2, n_mid=2, n_bottom=2,
                        min_headings=2, exclude_ids=None):
    df = rank_df.copy()
    df['image_name'] = df['image_name'].apply(
        lambda x: os.path.basename(str(x).strip())
    )
    df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')

    sorted_agg = agg_df.sort_values(
        'score_weighted', ascending=False
    ).reset_index(drop=True)

    sorted_agg = sorted_agg[sorted_agg['n_headings'] >= min_headings]
    if exclude_ids:
        sorted_agg = sorted_agg[~sorted_agg['point_id'].isin(
            [str(x) for x in exclude_ids]
        )]

    n_total    = len(sorted_agg)
    top_ids    = sorted_agg.head(n_top)['point_id'].tolist()
    bottom_ids = sorted_agg.tail(n_bottom).sort_values(
        'score_weighted'
    )['point_id'].tolist()
    mid_start  = n_total // 2 - n_mid // 2
    mid_ids    = sorted_agg.iloc[mid_start:mid_start + n_mid]['point_id'].tolist()

    print(f'  Top {n_top} (safest)      : {top_ids}')
    print(f'  Mid {n_mid} (medium)      : {mid_ids}')
    print(f'  Bottom {n_bottom} (dangerous) : {bottom_ids}')

    results = []
    for pid, category, color in (
        [(p, 'High safety',   '#1a9850') for p in top_ids]    +
        [(p, 'Medium safety', '#f4a42c') for p in mid_ids]    +
        [(p, 'Low safety',    '#d73027') for p in bottom_ids]
    ):
        grp = df[df['point_id'] == pid]
        if len(grp) == 0:
            continue
        best     = grp.loc[grp[sigma_col].idxmin()]
        loc_info = agg_df[agg_df['point_id'] == pid].iloc[0]
        lat = loc_info['lat']
        lng = loc_info['lng']
        print(f'    Geocoding point {pid} ({lat:.4f}, {lng:.4f})...')
        location_name = reverse_geocode(lat, lng)
        time.sleep(1.1)
        results.append({
            'point_id'     : pid,
            'image_name'   : best['image_name'],
            'mu'           : loc_info['score_weighted'],
            'sigma'        : best[sigma_col],
            'score_norm'   : loc_info['score_norm'],
            'category'     : category,
            'color'        : color,
            'location_name': location_name,
            'lat'          : lat,
            'lng'          : lng,
        })
    return results


def plot_streetview_examples(rank_df, mu_col, sigma_col,
                              agg_df, source_label, output_path,
                              n_top=2, n_mid=2, n_bottom=2,
                              min_headings=2, exclude_ids=None):

    print(f'\n{source_label} — selecting locations...')
    examples = get_example_images(
        rank_df, mu_col, sigma_col, agg_df,
        n_top, n_mid, n_bottom, min_headings, exclude_ids
    )
    n_cols = len(examples)

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))
    fig.patch.set_facecolor('white')
    fig.suptitle(
        f'{source_label}',
        fontsize=11, fontweight='bold', fontfamily=FONT_FAMILY, y=1.04
    )

    for col_idx, (ex, ax) in enumerate(zip(examples, axes)):
        ax.set_facecolor('white')
        try:
            pil_img = fetch_image(ex['image_name'])
            ax.imshow(pil_img)
        except Exception as e:
            ax.text(0.5, 0.5, f'Could not load\n{str(e)[:50]}',
                    ha='center', va='center',
                    transform=ax.transAxes, fontsize=7, color='red')
            ax.set_facecolor('#f8f8f8')

        for spine in ax.spines.values():
            spine.set_edgecolor(ex['color'])
            spine.set_linewidth(4)

        ax.text(0.5, 1.01, ex['category'],
                transform=ax.transAxes,
                ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                color=ex['color'],
                fontfamily=FONT_FAMILY)

        heading = ex['image_name'].replace('.jpg', '').split('_')[-1]
        caption = (
            f'{ex["location_name"]}\n'
            f'Heading {heading}\u00b0  |  '
            f'TrueSkill \u03bc = {ex["mu"]:.3f}  |  '
            f'\u03c3 = {ex["sigma"]:.3f}'
        )
        ax.set_xlabel(
            caption, fontsize=7.5,
            fontfamily=FONT_FAMILY,
            labelpad=8, color='#333333'
        )
        ax.set_xticks([])
        ax.set_yticks([])

    for div in [n_top / n_cols, (n_top + n_mid) / n_cols]:
        fig.add_artist(
            plt.Line2D(
                [div, div], [0.08, 0.95],
                transform=fig.transFigure,
                color='#aaaaaa', linewidth=1.5, linestyle='--'
            )
        )

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: {output_path}')


# ── exclude bad locations ─────────────────────────────────────────────────────
GPT_EXCLUDE   = ['4140', '2245']
HUMAN_EXCLUDE = ['2028', '4577']

# ── run for GPT ───────────────────────────────────────────────────────────────
print('Building GPT street view examples...')
plot_streetview_examples(
    rank_df      = gpt_df,
    mu_col       = 'gpt_mu',
    sigma_col    = 'gpt_sigma',
    agg_df       = gpt_agg,
    source_label = 'Panel C - GPT-5.2 synthetic votes',
    output_path  = os.path.join(OUTPUT_DIR, 'streetview_examples_gpt.pdf'),
    n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=GPT_EXCLUDE
)

# ── run for Human ─────────────────────────────────────────────────────────────
print('Building Human street view examples...')
plot_streetview_examples(
    rank_df      = human_df,
    mu_col       = 'human_mu',
    sigma_col    = 'human_sigma',
    agg_df       = human_agg,
    source_label = 'Panel B - Human crowdsourced votes',
    output_path  = os.path.join(OUTPUT_DIR, 'streetview_examples_human.pdf'),
    n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=HUMAN_EXCLUDE
)

# ── store examples as variables for combined figure ───────────────────────────
print('\nStoring examples for combined figure...')
examples_human = get_example_images(
    rank_df=human_df, mu_col='human_mu', sigma_col='human_sigma',
    agg_df=human_agg, n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=HUMAN_EXCLUDE
)
examples_gpt = get_example_images(
    rank_df=gpt_df, mu_col='gpt_mu', sigma_col='gpt_sigma',
    agg_df=gpt_agg, n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=GPT_EXCLUDE
)
print(f'\nexamples_human : {len(examples_human)} locations ready')
print(f'examples_gpt   : {len(examples_gpt)} locations ready')

## Map high safety and least safe images (Figure 6)

In [ ]:
import copy
import geopandas as gpd
import contextily as ctx
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
from matplotlib.patches import PathPatch
from matplotlib.path import Path
from shapely.geometry import Point, box
from shapely.ops import unary_union
import numpy as np
import requests as req
import io
import os
import time
import pandas as pd

mpl.rcParams['patch.antialiased'] = False

CARTO_API_KEY = '...'

def get_carto_basemap(base_provider):
    provider = copy.deepcopy(base_provider)
    provider['url'] = provider['url'] + f'?key={CARTO_API_KEY}'
    return provider

SAFETY_COLORS = {
    'High safety'  : '#1a7837',
    'Medium safety': '#e08214',
    'Low safety'   : '#b2182b',
}
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
DPI = 600
OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_NAME_LEN = 28


def truncate(name, maxlen=MAX_NAME_LEN):
    return name if len(name) <= maxlen else name[:maxlen - 1] + '\u2026'


def geom_to_path(geom):
    def ring_to_verts(ring):
        coords = np.array(ring.coords)
        return coords, [Path.MOVETO] + [Path.LINETO] * (len(coords) - 2) + [Path.CLOSEPOLY]
    verts, codes = [], []
    geoms = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for g in geoms:
        v, c = ring_to_verts(g.exterior)
        verts.extend(v); codes.extend(c)
        for interior in g.interiors:
            v, c = ring_to_verts(interior)
            verts.extend(v); codes.extend(c)
    return Path(np.array(verts), codes)


BOUNDARY_CACHE_PATH = os.path.join(OUTPUT_DIR, 'london_boundary_cache.geojson')

def load_london_boundary():
    try:
        if os.path.exists(BOUNDARY_CACHE_PATH):
            gdf_raw = gpd.read_file(BOUNDARY_CACHE_PATH)
            print('London boundary loaded from local cache.')
        else:
            url = ('https://cdn.jsdelivr.net/gh/martinjc/UK-GeoJSON@master/'
                   'json/administrative/eng/lad.json')
            resp = req.get(url, timeout=20)
            resp.raise_for_status()
            gdf_raw = gpd.read_file(io.BytesIO(resp.content))
            gdf_raw.to_file(BOUNDARY_CACHE_PATH, driver='GeoJSON')
            print('London boundary downloaded via jsDelivr and cached.')
        return gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].dissolve().to_crs('EPSG:3857')
    except Exception as e:
        print(f'Boundary failed: {e}')
        return None


london_boundary = load_london_boundary()
if london_boundary is None:
    raise RuntimeError('London boundary failed to load — cannot build an accurate map without it.')

SAFETY_CLASSES = ['High safety', 'Medium safety', 'Low safety']

def select_one_per_class(examples):
    selected = {}
    for ex in examples:
        cat = ex['category']
        if cat not in selected:
            selected[cat] = ex
    return [selected[cat] for cat in SAFETY_CLASSES if cat in selected]

human_sel = select_one_per_class(examples_human)
gpt_sel   = select_one_per_class(examples_gpt)

GEOCODE_CACHE_PATH = os.path.join(OUTPUT_DIR, 'reverse_geocode_cache.csv')

def load_geocode_cache():
    if os.path.exists(GEOCODE_CACHE_PATH):
        df = pd.read_csv(GEOCODE_CACHE_PATH)
        return dict(zip(df['point_id'], df['location_name']))
    return {}

def save_geocode_cache(cache_dict):
    pd.DataFrame(
        [{'point_id': k, 'location_name': v} for k, v in cache_dict.items()]
    ).to_csv(GEOCODE_CACHE_PATH, index=False)

def reverse_geocode_nominatim(lat, lng, point_id, cache):
    if point_id in cache:
        return cache[point_id]
    url = 'https://nominatim.openstreetmap.org/reverse'
    params = {'lat': lat, 'lon': lng, 'format': 'jsonv2'}
    headers = {'User-Agent': 'cycling-safety-research/1.0 (LSE academic project)'}
    resp = req.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    addr = data.get('address', {})
    road = addr.get('road', '')
    area = addr.get('suburb') or addr.get('town') or addr.get('city_district') or addr.get('borough', '')
    name = f'{road}, {area}'.strip(', ') if (road or area) else data.get('display_name', 'Unknown location')
    cache[point_id] = name
    save_geocode_cache(cache)
    time.sleep(1)
    return name

geocode_cache = load_geocode_cache()

HIGH_SAFETY_OVERRIDE = {
    'Human': {'point_id': '2002', 'image_name': '2002_90.jpg'},
    'GPT':   {'point_id': '3392', 'image_name': '3392_270.jpg'},
}

def apply_high_safety_override(sel_list, override, plot_df, cache):
    for ex in sel_list:
        if ex['category'] == 'High safety':
            row = plot_df[plot_df['point_id'] == override['point_id']].iloc[0]
            ex['image_name']    = override['image_name']
            ex['lat']           = row['lat']
            ex['lng']           = row['lng']
            ex['mu']            = row['score_weighted']
            ex['location_name'] = reverse_geocode_nominatim(
                row['lat'], row['lng'], override['point_id'], cache
            )
    return sel_list

human_sel = apply_high_safety_override(human_sel, HIGH_SAFETY_OVERRIDE['Human'], human_plot, geocode_cache)
gpt_sel   = apply_high_safety_override(gpt_sel,   HIGH_SAFETY_OVERRIDE['GPT'],   gpt_plot,   geocode_cache)

for i, ex in enumerate(human_sel):
    ex['label'] = f'H{i+1}'
for i, ex in enumerate(gpt_sel):
    ex['label'] = f'G{i+1}'

# ── Match Figure 7's exact headings for locations needing detection coverage ──
for ex in human_sel:
    if ex['label'] == 'H2':
        ex['image_name'] = '1334_90.jpg'
for ex in gpt_sel:
    if ex['label'] == 'G3':
        ex['image_name'] = '4561_90.jpg'
    if ex['label'] == 'G2':
        ex['image_name'] = '4163_180.jpg'

print("Confirmed selections after override:")
for ex in human_sel + gpt_sel:
    print(f"  {ex['label']}: {ex['image_name']} | {ex['location_name']} | mu={ex['mu']:.2f}")

all_examples = human_sel + gpt_sel
locs_df = gpd.GeoDataFrame(
    [{'lat'     : ex['lat'],
      'lng'     : ex['lng'],
      'category': ex['category'],
      'label'   : ex['label'],
      'source'  : 'Human' if ex['label'].startswith('H') else 'GPT'}
     for ex in all_examples],
    geometry=[Point(ex['lng'], ex['lat']) for ex in all_examples],
    crs='EPSG:4326'
).to_crs('EPSG:3857')

ZOOM_PADDING = 0.08
bounds = london_boundary.total_bounds
x_range = bounds[2] - bounds[0]
y_range = bounds[3] - bounds[1]
xlim = (bounds[0] - x_range * ZOOM_PADDING, bounds[2] + x_range * ZOOM_PADDING)
ylim = (bounds[1] - y_range * ZOOM_PADDING, bounds[3] + y_range * ZOOM_PADDING)

FIG_W = 20
FIG_H = 14

fig = plt.figure(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')

outer_gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 0.04], hspace=0.06)
inner_gs = gridspec.GridSpecFromSubplotSpec(
    3, 3, subplot_spec=outer_gs[0],
    width_ratios=[3.5, 1, 1], height_ratios=[1, 1, 1],
    hspace=0.25, wspace=0.04
)

ax_map = fig.add_subplot(inner_gs[:, 0])
ax_map.set_facecolor('white')
ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)
ax_map.set_aspect('equal')
ax_map.set_anchor('E')

ctx.add_basemap(ax_map, crs='EPSG:3857',
                source=get_carto_basemap(ctx.providers.CartoDB.Positron),
                zoom='auto', alpha=0.9)
ax_map.set_xlim(xlim)
ax_map.set_ylim(ylim)

london_u   = unary_union(london_boundary.geometry.values)
mask_geom  = box(xlim[0], ylim[0], xlim[1], ylim[1]).difference(london_u)
mask_patch = PathPatch(geom_to_path(mask_geom), fc='white', ec='none', alpha=1.0, zorder=4)
ax_map.add_patch(mask_patch)
london_boundary.boundary.plot(
    ax=ax_map, color='#333333', linewidth=1.0, zorder=5
)

MARKER_MAP = {'Human': 'o', 'GPT': '*'}
MARKER_SIZE = {'Human': 160, 'GPT': 260}

for _, row in locs_df.iterrows():
    color  = SAFETY_COLORS[row['category']]
    marker = MARKER_MAP[row['source']]
    msize  = MARKER_SIZE[row['source']]
    ax_map.scatter(
        row.geometry.x, row.geometry.y,
        color=color, marker=marker,
        s=msize, zorder=6, edgecolors='white', linewidths=1.0
    )
    ax_map.annotate(
        row['label'],
        xy=(row.geometry.x, row.geometry.y),
        xytext=(7, 7), textcoords='offset points',
        fontsize=7.5, fontweight='bold',
        color='#222222', zorder=7
    )

ax_map.set_axis_off()

fig.canvas.draw()
for col_idx, (col_label, col_pos) in enumerate([
    ('Human crowdsourced', inner_gs[0, 1]),
    ('GPT-5.2 synthetic',  inner_gs[0, 2]),
]):
    pos = col_pos.get_position(fig)
    fig.text(
        (pos.x0 + pos.x1) / 2, pos.y1 + 0.01, col_label,
        fontsize=9, fontweight='bold', ha='center', va='bottom', color='#222222'
    )

for row_idx, cat in enumerate(SAFETY_CLASSES):
    color        = SAFETY_COLORS[cat]
    human_ex     = human_sel[row_idx]
    gpt_ex       = gpt_sel[row_idx]

    for col_idx, ex in enumerate([human_ex, gpt_ex]):
        ax = fig.add_subplot(inner_gs[row_idx, col_idx + 1])
        ax.set_facecolor('white')
        try:
            pil_img = fetch_image(ex['image_name'])
            ax.imshow(pil_img)
        except Exception as e:
            ax.text(0.5, 0.5, f'Could not load\n{str(e)[:40]}',
                    ha='center', va='center',
                    transform=ax.transAxes, fontsize=6, color='red')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3.5)
        caption = (f'{ex["label"]} — {truncate(ex["location_name"])}'
                   f'\n\u03bc = {ex["mu"]:.1f}')
        ax.set_xlabel(caption, fontsize=7.5, labelpad=5, color='#333333')
        ax.set_xticks([])
        ax.set_yticks([])

    ax_lbl = fig.add_subplot(inner_gs[row_idx, 2])
    ax_lbl.set_axis_off()

fig.canvas.draw()
for row_idx, cat in enumerate(SAFETY_CLASSES):
    color   = SAFETY_COLORS[cat]
    pos     = inner_gs[row_idx, 2].get_position(fig)
    fig.text(
        pos.x1 + 0.005, (pos.y0 + pos.y1) / 2, cat,
        fontsize=8.5, fontweight='bold', color=color, va='center', ha='left', rotation=270
    )

ax_legend = fig.add_subplot(outer_gs[1])
ax_legend.set_axis_off()

legend_elements = [
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='#b2182b', markersize=9, label='Low safety — Human (circle)'),
    mlines.Line2D([0], [0], marker='*', color='w', markerfacecolor='#b2182b', markersize=12, label='Low safety — GPT-5.2 (star)'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='#e08214', markersize=9, label='Medium safety — Human (circle)'),
    mlines.Line2D([0], [0], marker='*', color='w', markerfacecolor='#e08214', markersize=12, label='Medium safety — GPT-5.2 (star)'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1a7837', markersize=9, label='High safety — Human (circle)'),
    mlines.Line2D([0], [0], marker='*', color='w', markerfacecolor='#1a7837', markersize=12, label='High safety — GPT-5.2 (star)'),
]
ax_legend.legend(
    handles=legend_elements, loc='center', ncol=3, fontsize=9,
    frameon=True, framealpha=0.95, edgecolor='#cccccc'
)

fig.suptitle(
    'Perceived cycling safety scores across Greater London',
    fontsize=13, fontweight='bold', y=1.002
)

out_path = os.path.join(OUTPUT_DIR, 'figure6_streetview_combined.pdf')
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
fig.savefig(out_path.replace('.pdf', '.png'), dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# Input for Cross Validation in the second stage

In [ ]:
# ── Borough and sub-region assignment for spatial cross-validation ────────────
# Outputs: point_borough_assignment.csv saved to OUTPUT_DIR.

import io as _io
import os
import requests
import geopandas as gpd
import pandas as pd

print('Building borough and sub-region assignment for spatial CV...')

# ── Load coordinates ──────────────────────────────────────────────────────────
coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

point_coords = (
    coords_df.groupby('point_id')[['actual_lat', 'actual_lng']]
    .first().reset_index()
    .rename(columns={'actual_lat': 'lat', 'actual_lng': 'lng'})
)
print(f'  Unique points: {len(point_coords):,}')

# ── Load London borough boundaries (cache-first, avoids GitHub rate limits) ───
BOUNDARY_CACHE_PATH = os.path.join(OUTPUT_DIR, 'london_boundary_cache.geojson')

if os.path.exists(BOUNDARY_CACHE_PATH):
    gdf_raw = gpd.read_file(BOUNDARY_CACHE_PATH)
    print('  Boundary loaded from local cache.')
else:
    url = ('https://cdn.jsdelivr.net/gh/martinjc/UK-GeoJSON@master/'
           'json/administrative/eng/lad.json')
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    gdf_raw = gpd.read_file(_io.BytesIO(resp.content))
    gdf_raw.to_file(BOUNDARY_CACHE_PATH, driver='GeoJSON')
    print('  Boundary downloaded via jsDelivr and cached.')

boroughs = (
    gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')]
    .copy()[['LAD13CD', 'LAD13NM', 'geometry']]
    .rename(columns={'LAD13CD': 'borough_code', 'LAD13NM': 'borough_name'})
    .set_crs('EPSG:4326', allow_override=True)
)
print(f'  Boroughs loaded: {len(boroughs)}')

# ── Official London Plan five sub-regions ─────────────────────────────────────
# Source: Greater London Authority, London Plan (2021)
SUBREGION_MAP = {
    # Central
    'City of London'           : 'Central',
    'Westminster'              : 'Central',
    'Kensington and Chelsea'   : 'Central',
    'Hammersmith and Fulham'   : 'Central',
    'Wandsworth'               : 'Central',
    'Lambeth'                  : 'Central',
    'Southwark'                : 'Central',
    # East
    'Hackney'                  : 'East',
    'Tower Hamlets'            : 'East',
    'Newham'                   : 'East',
    'Barking and Dagenham'     : 'East',
    'Havering'                 : 'East',
    'Redbridge'                : 'East',
    'Waltham Forest'           : 'East',
    # North
    'Barnet'                   : 'North',
    'Enfield'                  : 'North',
    'Haringey'                 : 'North',
    'Camden'                   : 'North',
    'Islington'                : 'North',
    # South
    'Bromley'                  : 'South',
    'Croydon'                  : 'South',
    'Sutton'                   : 'South',
    'Merton'                   : 'South',
    'Kingston upon Thames'     : 'South',
    'Richmond upon Thames'     : 'South',
    'Lewisham'                 : 'South',
    'Greenwich'                : 'South',
    'Bexley'                   : 'South',
    # West
    'Ealing'                   : 'West',
    'Hounslow'                 : 'West',
    'Hillingdon'               : 'West',
    'Harrow'                   : 'West',
    'Brent'                    : 'West',
}

# ── Spatial join: assign each point to a borough ─────────────────────────────
points_gdf = gpd.GeoDataFrame(
    point_coords,
    geometry=gpd.points_from_xy(point_coords['lng'], point_coords['lat']),
    crs='EPSG:4326'
)

# Primary join: within
joined = gpd.sjoin(
    points_gdf,
    boroughs[['borough_code', 'borough_name', 'geometry']],
    how='left',
    predicate='within'
)
joined = (
    joined[['point_id', 'lat', 'lng', 'borough_code', 'borough_name']]
    .drop_duplicates(subset='point_id')
    .reset_index(drop=True)
)

# Secondary join: nearest neighbour for unmatched points
unmatched_mask = joined['borough_name'].isna()
n_unmatched    = unmatched_mask.sum()
print(f'  Points matched (within)  : {(~unmatched_mask).sum():,}')
print(f'  Points unmatched         : {n_unmatched:,}')

if n_unmatched > 0:
    unmatched_gdf = points_gdf[
        points_gdf['point_id'].isin(joined.loc[unmatched_mask, 'point_id'])
    ].copy()
    nearest = gpd.sjoin_nearest(
        unmatched_gdf,
        boroughs[['borough_code', 'borough_name', 'geometry']],
        how='left'
    )[['point_id', 'borough_code', 'borough_name']].drop_duplicates(subset='point_id')
    for _, row in nearest.iterrows():
        mask = joined['point_id'] == row['point_id']
        joined.loc[mask, 'borough_code'] = row['borough_code']
        joined.loc[mask, 'borough_name'] = row['borough_name']
    print(f'  Unmatched assigned by nearest neighbour: {n_unmatched}')

# ── Add sub-region label ──────────────────────────────────────────────────────
joined['subregion'] = joined['borough_name'].map(SUBREGION_MAP)
n_no_subregion = joined['subregion'].isna().sum()
if n_no_subregion > 0:
    print(f'  WARNING: {n_no_subregion} points have no sub-region assigned.')
    print(joined[joined['subregion'].isna()][['point_id', 'borough_name']])

# ── Diagnostics ───────────────────────────────────────────────────────────────
print()
print('Sub-region distribution:')
sr_counts = joined['subregion'].value_counts().sort_index()
for sr, cnt in sr_counts.items():
    print(f'  {sr:<10}: {cnt:,} points')
print(f'  Total    : {len(joined):,} points')
print()
print('Borough distribution by sub-region:')
for sr in sorted(SUBREGION_MAP.values()):
    if sr not in sr_counts.index:
        continue
    boroughs_in_sr = joined[joined['subregion'] == sr]['borough_name'].value_counts()
    print(f'  {sr}:')
    for b, c in boroughs_in_sr.items():
        print(f'    {b:<35}: {c:,}')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = os.path.join(OUTPUT_DIR, 'point_borough_assignment.csv')
joined[['point_id', 'lat', 'lng', 'borough_code', 'borough_name', 'subregion']].to_csv(
    out_path, index=False
)
print(f'\nSaved: {out_path}')
print('Upload this file to your Kaggle dataset (insumos2) before running the stage 2 notebook.')

# Figure 7: Map and images for Object detection

In [ ]:
## Setup: fetch functions + path constants + populate `fetched`
import io
import json
import numpy as np
from PIL import Image

PANOPTIC_JSON_DIR   = 'PanopticSegmentation/json/'
PANOPTIC_MASK_DIR   = 'PanopticSegmentation/'
MASK_PATH_TEMPLATE  = 'PanopticSegmentation/masks/{stem}.png'
YOLO_PREFIX         = 'YoloV10/conf_0.5/images/'


def fetch_bytes(blob_path):
    return container_client.download_blob(blob_path, timeout=30).readall()

def blob_exists(blob_path):
    try:
        container_client.get_blob_client(blob_path).get_blob_properties()
        return True
    except Exception:
        return False

def fetch_json_for_image(image_name):
    stem      = image_name.replace('.jpg', '')
    json_path = PANOPTIC_JSON_DIR + stem + '.json'
    if not blob_exists(json_path):
        json_path = PANOPTIC_MASK_DIR + stem + '.json'
    raw = fetch_bytes(json_path).decode('utf-8')
    return json.loads(raw)

def fetch_mask_as_rgb(image_name, data):
    stem      = image_name.replace('.jpg', '')
    mask_path = MASK_PATH_TEMPLATE.format(stem=stem)
    mask_data = fetch_bytes(mask_path)
    mask_img  = Image.open(io.BytesIO(mask_data))

    if mask_img.mode in ('RGB', 'RGBA'):
        mask_arr = np.array(mask_img.convert('RGB'))
        seg_ids  = (mask_arr[:, :, 0].astype(np.int32)
                    + mask_arr[:, :, 1].astype(np.int32) * 256
                    + mask_arr[:, :, 2].astype(np.int32) * 256 * 256)
    else:
        seg_ids = np.array(mask_img.convert('I')).astype(np.int32)

    id_to_color = {}
    for seg in data.get('segments_info', []):
        label = seg['label_name']
        color = PANOPTIC_COLORS.get(label, [128, 128, 128])
        id_to_color[seg['id']] = color

    h, w = seg_ids.shape
    rgb  = np.zeros((h, w, 3), dtype=np.uint8)
    for seg_id, color in id_to_color.items():
        mask = seg_ids == seg_id
        rgb[mask] = color

    return Image.fromarray(rgb)


print('Fetch functions and constants defined.')
print()

# ── Populate `fetched` for real this time ──────────────────────────────────────
print('Fetching images, masks and detection overlays from Azure...')
fetched = {}

all_images = list(set(
    list(HUMAN_IMAGES_FIG7.values()) + list(GPT_IMAGES_FIG7.values())
))

for img_name in all_images:
    print(f'  {img_name}', end=' ')
    try:
        j    = fetch_json_for_image(img_name)
        orig = fetch_image(img_name)
        mask = fetch_mask_as_rgb(img_name, j)
        det_path = YOLO_PREFIX + img_name
        try:
            det_data = fetch_bytes(det_path)
            det_img  = Image.open(io.BytesIO(det_data)).convert('RGB')
        except Exception:
            det_img  = None
            print('(no detection)', end=' ')
        fetched[img_name] = {'orig': orig, 'mask': mask, 'json': j, 'det': det_img}
        print('OK')
    except Exception as e:
        print(f'FAILED: {e}')

print()
print(f'fetched now has {len(fetched)} entries.')

In [ ]:
# ── Figure 7: Panoptic segmentation masks and object detection overlays ───────
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import os

SAFETY_CLASSES_ORDER = ['High safety', 'Medium safety', 'Low safety']
PANELS = [
    ('Panel A — Human crowdsourced votes', image_lookup_fig7['Human']),
    ('Panel B — GPT-5.2 synthetic votes',  image_lookup_fig7['GPT']),
]
ROW_LABELS = ['Original', 'Segmentation\nmask', 'Object\ndetection']
ROW_KEYS   = ['orig', 'mask', 'det']

FIG_W = 18
FIG_H = 27

fig = plt.figure(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(
    3, 1, figure=fig,
    height_ratios=[3, 3, 0.12],
    hspace=0.05
)

for panel_idx, (panel_label, lookup) in enumerate(PANELS):
    inner = gridspec.GridSpecFromSubplotSpec(
        4, 4,
        subplot_spec=outer[panel_idx],
        height_ratios=[0.45, 1, 1, 1],
        hspace=0.04,
        wspace=0.08,
        width_ratios=[0.35, 1, 1, 1]
    )

    for row_idx in range(3):
        grid_row = row_idx + 1
        for col_idx, cat in enumerate(SAFETY_CLASSES_ORDER):
            ax = fig.add_subplot(inner[grid_row, col_idx + 1])
            ax.set_facecolor('white')
            ax.set_xticks([])
            ax.set_yticks([])

            img_name = lookup[cat][0]
            entry    = fetched.get(img_name, {})
            img_data = entry.get(ROW_KEYS[row_idx])
            color    = SAFETY_COLORS[cat]

            if img_data is not None:
                ax.imshow(img_data, aspect='auto')
            else:
                ax.text(0.5, 0.5,
                        'Not available' if ROW_KEYS[row_idx] == 'det' else 'Load failed',
                        ha='center', va='center',
                        transform=ax.transAxes,
                        fontsize=8, color='#888888')

            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(1.5)

        ax_lbl = fig.add_subplot(inner[grid_row, 0])
        ax_lbl.set_axis_off()
        ax_lbl.text(
            0.9, 0.5,
            ROW_LABELS[row_idx],
            fontsize=11, fontweight='bold', fontstyle='italic',
            color='#444444',
            ha='right', va='center',
            transform=ax_lbl.transAxes
        )

    fig.canvas.draw()

    for col_idx, cat in enumerate(SAFETY_CLASSES_ORDER):
        pos = inner[0, col_idx + 1].get_position(fig)
        color = SAFETY_COLORS[cat]
        fig.text(
            (pos.x0 + pos.x1) / 2,
            pos.y0 + (pos.y1 - pos.y0) * 0.25,
            cat,
            fontsize=13, fontweight='bold',
            color=color, ha='center', va='bottom'
        )

    pos_left = inner[0, 1].get_position(fig)
    fig.text(
        pos_left.x0,
        pos_left.y0 + (pos_left.y1 - pos_left.y0) * 0.65,
        panel_label,
        fontsize=14, fontweight='bold',
        color='#222222', ha='left', va='bottom'
    )

ax_legend = fig.add_subplot(outer[2])
ax_legend.set_axis_off()

legend_classes = [
    'Road', 'Sidewalk', 'Building', 'Vegetation', 'Sky',
    'Car', 'Person', 'Bicycle', 'Bike Lane', 'Terrain',
    'Wall', 'Fence', 'Pole', 'Traffic Sign (Front)', 'Bus',
]

missing_classes = [cls for cls in legend_classes if cls not in PANOPTIC_COLORS]
if missing_classes:
    raise KeyError(
        f"These legend classes are missing from PANOPTIC_COLORS: {missing_classes}. "
        f"Add them to PANOPTIC_COLORS or drop them from legend_classes."
    )

legend_handles = [
    mpatches.Patch(
        color=[c / 255 for c in PANOPTIC_COLORS[cls]],
        label=cls
    )
    for cls in legend_classes
]
ax_legend.legend(
    handles=legend_handles,
    loc='center', ncol=8,
    fontsize=11, frameon=True,
    framealpha=0.95, edgecolor='#cccccc',
    columnspacing=1.0, handlelength=1.5
)

# ── Save ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DPI = 600

out_png = os.path.join(OUTPUT_DIR, 'figure_segmentation_masks.png')
out_pdf = os.path.join(OUTPUT_DIR, 'figure_segmentation_masks.pdf')

fig.savefig(out_png, dpi=DPI, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print(f'Saved PNG: {out_png}')
print(f'Saved PDF: {out_pdf}')

# Figure for Appendix

In [ ]:
# ── Appendix C: Average perceived cycling safety score by London borough ──────
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np
import os

DATA_DIR   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
EXPORT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\paper_exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# ── Load point-to-borough assignment (built earlier in this notebook) ─────────
borough_assign = pd.read_csv(os.path.join(DATA_DIR, 'point_borough_assignment.csv'))
borough_assign['point_id'] = borough_assign['point_id'].astype(str)

# ── Compute per-borough average normalised score AND point count ──────────────
def borough_scores(plot_df, borough_assign):
    df = plot_df.copy()
    df['point_id'] = df['point_id'].astype(str)
    merged = df.merge(borough_assign[['point_id', 'borough_name']], on='point_id', how='inner')
    merged['score_norm'] = (
        (merged['score_weighted'] - merged['score_weighted'].min()) /
        (merged['score_weighted'].max() - merged['score_weighted'].min())
    )
    agg = merged.groupby('borough_name')['score_norm'].agg(['mean', 'count']).reset_index()
    agg.columns = ['borough_name', 'avg_score', 'n_points']
    return agg

human_borough_scores = borough_scores(human_plot, borough_assign)
gpt_borough_scores   = borough_scores(gpt_plot, borough_assign)

# ── Join scores onto the borough boundaries GeoDataFrame ──────────────────────
boroughs_human = boroughs.merge(human_borough_scores, on='borough_name', how='left')
boroughs_gpt    = boroughs.merge(gpt_borough_scores,   on='borough_name', how='left')

# ── Determine shared colour scale: min/median/max across BOTH sources ─────────
all_scores = pd.concat([human_borough_scores['avg_score'], gpt_borough_scores['avg_score']])
vmin    = all_scores.min()
vmax    = all_scores.max()
vcenter = all_scores.median()

print(f"Colour scale — min: {vmin:.3f}, median: {vcenter:.3f}, max: {vmax:.3f}")

norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
cmap = plt.get_cmap('RdYlGn')

# ── Figure ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8.5))
fig.patch.set_facecolor('white')

panels = [
    (axes[0], boroughs_human, 'Panel A — Human crowdsourced votes'),
    (axes[1], boroughs_gpt,   'Panel B — GPT-5.2 synthetic votes'),
]

for ax, gdf, title in panels:
    ax.set_facecolor('white')

    no_data  = gdf[gdf['avg_score'].isna()]
    has_data = gdf[gdf['avg_score'].notna()]

    if not no_data.empty:
        no_data.plot(ax=ax, color='#d9d9d9', edgecolor='#666666', linewidth=0.5)

    has_data.plot(
        ax=ax, column='avg_score', cmap=cmap, norm=norm,
        edgecolor='#444444', linewidth=0.5
    )

    # Borough name + point count, with a white halo so text is readable
    # regardless of the underlying polygon colour.
    for _, row in gdf.iterrows():
        centroid = row.geometry.centroid
        n_label = f"n={int(row['n_points'])}" if pd.notna(row.get('n_points')) else "n/a"
        txt = ax.annotate(
            f"{row['borough_name']}\n{n_label}",
            xy=(centroid.x, centroid.y),
            ha='center', va='center',
            fontsize=6.5, fontweight='bold', color='#111111',
            linespacing=1.3
        )
        txt.set_path_effects([
            path_effects.Stroke(linewidth=2.2, foreground='white'),
            path_effects.Normal()
        ])

    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_axis_off()

# ── Figure title — given explicit vertical room so it never gets clipped ──────
fig.suptitle('Average perceived cycling safety score by London borough',
             fontsize=13, fontweight='bold', y=0.97)

# ── Shared colourbar ────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.35, 0.09, 0.3, 0.025])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label(
    f'Average normalised safety score  '
    f'(min={vmin:.2f}, median={vcenter:.2f}, max={vmax:.2f})',
    fontsize=8
)
cbar.ax.tick_params(labelsize=7)

# Reserve explicit top and bottom margins instead of relying only on
# suptitle's y= and bbox_inches='tight', which was clipping the title.
plt.subplots_adjust(top=0.86, bottom=0.11)

png_path = os.path.join(EXPORT_DIR, 'figureC1_borough_choropleth.png')
pdf_path = os.path.join(EXPORT_DIR, 'figureC1_borough_choropleth.pdf')
fig.savefig(png_path, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
print(f'Saved: {png_path}')
print(f'Saved: {pdf_path}')

plt.show()